In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import evaluate, summary, time_inference, time_inference_torch
from gps_ids import protocol
from gps_ids.results import append_validated

import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import time
import os

In [2]:
# Constants
RANDOM_STATE = 1082
TRAIN_SIZE = 0.8
RESULTS_CSV = "../../../UAVATTACK_Results.csv"

In [ ]:
feature_cols_1 = ['vdop', 'ax', 'hdop', 'cog_rad', 'jamming_indicator', 'terrain_alt_valid', 'dist_bottom_valid', 'terrain_alt', 'lat_x', 'lat_y']
feature_cols_2 = ['lat_y', 'vel_e_m_s', 'vel_n_m_s', 'c_variance_rad', 'hdop', 's_variance_m_s', 'q[0]', 'vel_m_s' ,'cog_rad', 'q[3]']

EXP1 = protocol.real_heldout("uav", features=feature_cols_1)
EXP2 = protocol.real_heldout("uav", features=feature_cols_2)
FEATURE_EXPS = {"Explainability Features": EXP1, "Xgboost Features": EXP2}

X1_train_scaled, X1_test_scaled = EXP1.X_train, EXP1.X_test
X2_train_scaled, X2_test_scaled = EXP2.X_train, EXP2.X_test
y_train, y_test = EXP1.y_train, EXP1.y_test


In [5]:
results = []

In [ ]:
def add_results(model, model_name, feature_set, x_test, y_test, train_time, params, num_epochs):
    exp = FEATURE_EXPS[feature_set]
    y_pred = model.predict(x_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    print(f"{model_name} ({feature_set}) Accuracy: {accuracy * 100:.2f}%")
    print(f"Classification report ({model_name}, {feature_set}):")
    print(classification_report(y_test, y_pred, zero_division=0))
    _row = evaluate(
        y_test,
        y_pred,
        model_name=f"{model_name} ({feature_set})",
        family="FeatureEngineering",
        y_proba=model.predict_proba(x_test) if hasattr(model, "predict_proba") else None,
        train_time_sec=train_time,
        num_epochs=num_epochs,
        parameters=str(params),
        timing=time_inference(model.predict, x_test),
        **exp.eval_kwargs(),
    )
    print(summary(_row))
    results.append(_row)


In [7]:
# XGBoost on both sets
model1 = XGBClassifier(
    objective='binary:logistic',
    random_state=RANDOM_STATE,
    eval_metric='mae',
)
start_time_1 = time.time()
model1.fit(X1_train_scaled, y_train)
train_time_1 = time.time() - start_time_1
add_results(model1, "XGBoost", "Explainability Features", X1_test_scaled, y_test, train_time_1, model1.get_params(), "N/A")

model2 = XGBClassifier(
    objective='binary:logistic',
    random_state=RANDOM_STATE,
    eval_metric='mae',
)
start_time_2 = time.time()
model2.fit(X2_train_scaled, y_train)
train_time_2 = time.time() - start_time_2
add_results(model2, "XGBoost", "Xgboost Features", X2_test_scaled, y_test, train_time_2, model2.get_params(), "N/A")

XGBoost (Explainability Features) Accuracy: 99.79%
Classification report (XGBoost, Explainability Features):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.98      0.99       398

    accuracy                           1.00      2898
   macro avg       1.00      0.99      1.00      2898
weighted avg       1.00      1.00      1.00      2898

XGBoost (Xgboost Features) Accuracy: 99.79%
Classification report (XGBoost, Xgboost Features):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.98      0.99       398

    accuracy                           1.00      2898
   macro avg       1.00      0.99      1.00      2898
weighted avg       1.00      1.00      1.00      2898



In [8]:
# MLPClassifier on both sets
mlp1 = MLPClassifier(
    hidden_layer_sizes=(100, 50, 25),
    activation='tanh',
    solver='adam',
    random_state=RANDOM_STATE,
    learning_rate='adaptive',
    early_stopping=True,
)
start_time_mlp1 = time.time()
mlp1.fit(X1_train_scaled, y_train)
train_time_mlp1 = time.time() - start_time_mlp1
add_results(mlp1, "MLP Classifier", "Explainability Features", X1_test_scaled, y_test, train_time_mlp1, mlp1.get_params(), mlp1.n_iter_)

mlp2 = MLPClassifier(
    hidden_layer_sizes=(100, 50, 25),
    activation='tanh',
    solver='adam',
    random_state=RANDOM_STATE,
    learning_rate='adaptive',
    early_stopping=True,
)
start_time_mlp2 = time.time()
mlp2.fit(X2_train_scaled, y_train)
train_time_mlp2 = time.time() - start_time_mlp2
add_results(mlp2, "MLP Classifier", "Xgboost Features", X2_test_scaled, y_test, train_time_mlp2, mlp2.get_params(), mlp2.n_iter_)

MLP Classifier (Explainability Features) Accuracy: 99.65%
Classification report (MLP Classifier, Explainability Features):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.97      0.99       398

    accuracy                           1.00      2898
   macro avg       1.00      0.99      0.99      2898
weighted avg       1.00      1.00      1.00      2898

MLP Classifier (Xgboost Features) Accuracy: 99.76%
Classification report (MLP Classifier, Xgboost Features):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.98      0.99       398

    accuracy                           1.00      2898
   macro avg       1.00      0.99      0.99      2898
weighted avg       1.00      1.00      1.00      2898



In [9]:
# Logistic Regression on both sets
logreg1 = LogisticRegression(
    solver='saga',
    max_iter=2000,
    random_state=RANDOM_STATE,
)
start_time_logreg1 = time.time()
logreg1.fit(X1_train_scaled, y_train)
train_time_logreg1 = time.time() - start_time_logreg1
add_results(logreg1, "Logistic Regression", "Explainability Features", X1_test_scaled, y_test, train_time_logreg1, logreg1.get_params(), "N/A")

logreg2 = LogisticRegression(
    solver='saga',
    max_iter=2000,
    random_state=RANDOM_STATE,
)
start_time_logreg2 = time.time()
logreg2.fit(X2_train_scaled, y_train)
train_time_logreg2 = time.time() - start_time_logreg2
add_results(logreg2, "Logistic Regression", "Xgboost Features", X2_test_scaled, y_test, train_time_logreg2, logreg2.get_params(), "N/A")

Logistic Regression (Explainability Features) Accuracy: 99.76%
Classification report (Logistic Regression, Explainability Features):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.98      0.99       398

    accuracy                           1.00      2898
   macro avg       1.00      0.99      0.99      2898
weighted avg       1.00      1.00      1.00      2898

Logistic Regression (Xgboost Features) Accuracy: 99.79%
Classification report (Logistic Regression, Xgboost Features):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.98      0.99       398

    accuracy                           1.00      2898
   macro avg       1.00      0.99      1.00      2898
weighted avg       1.00      1.00      1.00      2898



In [10]:
# Random Forest on both sets
rf1 = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
start_time_rf1 = time.time()
rf1.fit(X1_train_scaled, y_train)
train_time_rf1 = time.time() - start_time_rf1
add_results(rf1, "Random Forest", "Explainability Features", X1_test_scaled, y_test, train_time_rf1, rf1.get_params(), "N/A")

rf2 = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
start_time_rf2 = time.time()
rf2.fit(X2_train_scaled, y_train)
train_time_rf2 = time.time() - start_time_rf2
add_results(rf2, "Random Forest", "Xgboost Features", X2_test_scaled, y_test, train_time_rf2, rf2.get_params(), "N/A")

Random Forest (Explainability Features) Accuracy: 99.86%
Classification report (Random Forest, Explainability Features):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.99      0.99       398

    accuracy                           1.00      2898
   macro avg       1.00      0.99      1.00      2898
weighted avg       1.00      1.00      1.00      2898

Random Forest (Xgboost Features) Accuracy: 99.90%
Classification report (Random Forest, Xgboost Features):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2500
           1       1.00      0.99      1.00       398

    accuracy                           1.00      2898
   macro avg       1.00      1.00      1.00      2898
weighted avg       1.00      1.00      1.00      2898



In [ ]:
append_validated(results, EXP1.results_csv)
print(f"Results appended to {EXP1.results_csv}")
